# Case outcomes — reviewer point 2 (ICPM 2027, submission 56)

> *Report how cases end (e.g., the share of canceled and completed cases) for DRL-DRL compared with the baselines.*

The question is whether DRL-DRL improves SLA compliance partly by **shortening cases / steering them to early terminal activities**.
For every log × policy × run we compute:

- **Events/case**: mean trace length.
- **End-activity distribution**: share of cases per last activity (sorted by `end_time`).
- **Canceled share** (AC_CRE only — the only log with explicit cancellation activities; BPIC12_W / BPIC17_W contain only `W_` work items, so no explicit outcome is observable).
- **Focus end share**: share of cases ending in the most common end activity of the original (train) log, fixed a priori.
- **TVD_end**: total variation distance between the end-activity distribution and the original log's (0 = identical, 1 = disjoint).

Mean ± 95% CI over the 10 simulation runs (t-distribution, same as `src/evaluation/metrics/functions/aggregation.py::mean_and_ci`).

Two outputs: an **internal table** (all four experimental conditions) and a **paper table** (flex condition, as in the submission).

In [1]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd

# Run from project root (notebook lives in analysis/)
if Path.cwd().name == "analysis":
    os.chdir("..")
assert Path("data/evaluation_results").exists(), Path.cwd()
OUT_DIR = Path("analysis")

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)

In [2]:
LOGS = {  # folder name -> label used in the paper
    "AcademicCredentials": "AC_CRE",
    "BPIC_2012": "BPIC12_W",
    "BPIC_2017": "BPIC17_W",
}

# condition folder -> {policy folder: display name}
CONDITIONS = {
    "flex":           {"DM-RR": "DM-RR", "RA-RR": "RA-RR", "DM-GR": "DM-GR", "DM-DRL": "DM-DRL", "DRL-DRL": "DRL-DRL"},
    "flex_kl_reg":    {"DRL-DRL": "DRL-DRL"},
    "noflex_matrix":  {"DRL-DRL-noflex": "DRL-DRL"},
    "no_flex_kl_reg": {"DRL-DRL": "DRL-DRL"},
}
PAPER_CONDITION = "flex"
PAPER_POLICIES = ["DM-RR", "DM-DRL", "DRL-DRL"]

# Explicit cancellation activities (only AC_CRE has them)
CANCEL_ACTIVITIES = {
    "AcademicCredentials": {"Notificacion estudiante cancelacion soli", "Cancelar Solicitud", "Cancelar curso"},
}

In [3]:
# Focus end activity = most common end activity in the ORIGINAL (train) log.
# Defined a priori from the reference log, independent of any policy's output.
def _last_activity_shares(path):
    df = pd.read_csv(path)
    df["activity"] = df["activity"].astype(str).str.strip()
    df = df.sort_values(["case_id", "end_time", "start_time"], kind="stable")
    return df.groupby("case_id")["activity"].last().value_counts(normalize=True)

FOCUS_END = {log: _last_activity_shares(f"data/logs/{log}/{log}_train.csv").idxmax() for log in LOGS}
FOCUS_END

{'AcademicCredentials': 'Notificacion estudiante cancelacion soli',
 'BPIC_2012': 'W_Afhandelen leads',
 'BPIC_2017': 'W_Validate application'}

In [4]:
def mean_and_ci(values, confidence=0.95):
    # Same as src/evaluation/metrics/functions/aggregation.py::mean_and_ci
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n < 2:
        return float(values.mean()), 0.0
    try:
        from scipy import stats
        t_val = stats.t.ppf((1 + confidence) / 2, n - 1)
    except ImportError:  # fallback: t_{0.975, n-1} table for small n
        t_val = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262}.get(n, 1.96)
    se = values.std(ddof=1) / np.sqrt(n)
    return float(values.mean()), float(t_val * se)


def load_log(path):
    df = pd.read_csv(path)
    df["activity"] = df["activity"].astype(str).str.strip()
    return df


def case_table(df):
    # One row per case: trace length and last activity (by end_time, ties by start_time).
    df = df.sort_values(["case_id", "end_time", "start_time"], kind="stable")
    g = df.groupby("case_id", sort=False)
    return pd.DataFrame({"length": g.size(), "end_activity": g["activity"].last()})


def run_metrics(cases, log, ref_end_dist):
    end_dist = cases["end_activity"].value_counts(normalize=True)
    acts = ref_end_dist.index.union(end_dist.index)
    tvd = 0.5 * (ref_end_dist.reindex(acts, fill_value=0) - end_dist.reindex(acts, fill_value=0)).abs().sum()
    m = {
        "n_cases": len(cases),
        "events_per_case": cases["length"].mean(),
        "median_events_per_case": cases["length"].median(),
        "single_event_share": (cases["length"] == 1).mean(),
        "focus_end_share": end_dist.get(FOCUS_END[log], 0.0),
        "tvd_end": tvd,
    }
    if log in CANCEL_ACTIVITIES:
        m["canceled_share"] = cases["end_activity"].isin(CANCEL_ACTIVITIES[log]).mean()
        m["completed_share"] = 1 - m["canceled_share"]
    return m, end_dist

## Compute per-run metrics

In [5]:
records, end_records = [], []

for log, label in LOGS.items():
    ref_cases = case_table(load_log(f"data/logs/{log}/{log}_train.csv"))
    ref_end = ref_cases["end_activity"].value_counts(normalize=True)

    # Original log (train = what the simulator is fitted on; test shown for reference)
    for split in ["train", "test"]:
        cases = case_table(load_log(f"data/logs/{log}/{log}_{split}.csv"))
        m, ed = run_metrics(cases, log, ref_end)
        name = f"Original ({split})"
        records.append({"condition": "original", "log": label, "policy": name, "run_id": 0, **m})
        end_records += [{"condition": "original", "log": label, "policy": name, "run_id": 0, "end_activity": a, "share": s} for a, s in ed.items()]

    for cond, policies in CONDITIONS.items():
        for folder, pol in policies.items():
            files = sorted(glob.glob(f"data/evaluation_results/{cond}/{log}/{folder}/simulated_logs/sim_run_*.csv"))
            if not files:
                print(f"[skip] {cond}/{log}/{folder}: no simulated logs")
                continue
            for i, f in enumerate(files, 1):
                cases = case_table(load_log(f))
                m, ed = run_metrics(cases, log, ref_end)
                records.append({"condition": cond, "log": label, "policy": pol, "run_id": i, **m})
                end_records += [{"condition": cond, "log": label, "policy": pol, "run_id": i, "end_activity": a, "share": s} for a, s in ed.items()]

runs = pd.DataFrame(records)
ends = pd.DataFrame(end_records)
runs.to_csv(OUT_DIR / "case_outcomes_runs.csv", index=False)
runs.groupby(["condition", "log", "policy"]).size().rename("n_runs").to_frame().T

condition   flex                                                                                                 flex_kl_reg                   no_flex_kl_reg                   noflex_matrix                          original                   \
log       AC_CRE                           BPIC12_W                           BPIC17_W                                AC_CRE BPIC12_W BPIC17_W         AC_CRE BPIC12_W BPIC17_W        AC_CRE BPIC12_W BPIC17_W          AC_CRE                    
policy    DM-DRL DM-GR DM-RR DRL-DRL RA-RR   DM-DRL DM-GR DM-RR DRL-DRL RA-RR   DM-DRL DM-GR DM-RR DRL-DRL RA-RR     DRL-DRL  DRL-DRL  DRL-DRL        DRL-DRL  DRL-DRL  DRL-DRL       DRL-DRL  DRL-DRL  DRL-DRL Original (test) Original (train)   
n_runs        10    10    10      10    10       10    10    10      10    10       10    10    10      10    10          10       10       10             10       10       10            10       10       10               1                1   

condition                                                                    
log              BPIC12_W                         BPIC17_W                   
policy    Original (test) Original (train) Original (test) Original (train)  
n_runs                  1                1               1                1

In [6]:
METRICS = ["n_cases", "events_per_case", "median_events_per_case", "single_event_share",
           "canceled_share", "completed_share", "focus_end_share", "tvd_end"]

def aggregate(df):
    rows = []
    for key, grp in df.groupby(["condition", "log", "policy"], sort=False):
        row = dict(zip(["condition", "log", "policy"], key))
        row["n_runs"] = len(grp)
        for m in METRICS:
            if m in grp and grp[m].notna().any():
                row[f"{m}_mean"], row[f"{m}_ci95"] = mean_and_ci(grp[m].dropna())
        rows.append(row)
    return pd.DataFrame(rows)

agg = aggregate(runs)
agg.to_csv(OUT_DIR / "case_outcomes_aggregated.csv", index=False)

def fmt(agg, metrics, pct=()):
    out = agg[["condition", "log", "policy"]].copy()
    for m in metrics:
        mean, ci = agg.get(f"{m}_mean"), agg.get(f"{m}_ci95")
        if mean is None:
            continue
        k = 100 if m in pct else 1
        out[m] = [("—" if pd.isna(a) else (f"{a*k:.2f}" if b == 0 or pd.isna(b) else f"{a*k:.2f} ± {b*k:.2f}")) for a, b in zip(mean, ci)]
    return out

## Internal table — all conditions

In [7]:
PCT = {"single_event_share", "canceled_share", "completed_share", "focus_end_share"}
cond_order = ["original"] + list(CONDITIONS)
pol_order = ["Original (train)", "Original (test)", "RA-RR", "DM-RR", "DM-GR", "DM-DRL", "DRL-DRL"]

internal = fmt(agg, ["events_per_case", "single_event_share", "canceled_share", "focus_end_share", "tvd_end"], pct=PCT)
internal["_c"] = internal["condition"].map(cond_order.index)
internal["_p"] = internal["policy"].map(pol_order.index)
internal = internal.sort_values(["log", "_c", "_p"]).drop(columns=["_c", "_p"]).reset_index(drop=True)
internal.to_csv(OUT_DIR / "case_outcomes_all.csv", index=False)
print("Shares in %. FOCUS_END:", FOCUS_END)
internal

Shares in %. FOCUS_END: {'AcademicCredentials': 'Notificacion estudiante cancelacion soli', 'BPIC_2012': 'W_Afhandelen leads', 'BPIC_2017': 'W_Validate application'}


,condition,log,policy,events_per_case,single_event_share,canceled_share,focus_end_share,tvd_end
0,original,AC_CRE,Original (train),4.89,1.51,54.52,51.51,0.00
1,original,AC_CRE,Original (test),4.49,0.75,53.27,48.74,0.20
2,flex,AC_CRE,RA-RR,6.96 ± 0.13,0.00,60.58 ± 0.98,60.58 ± 0.98,0.20 ± 0.01
3,flex,AC_CRE,DM-RR,5.18 ± 0.06,0.00,50.98 ± 2.28,50.98 ± 2.28,0.14 ± 0.01
4,flex,AC_CRE,DM-GR,5.21 ± 0.04,0.00,50.30 ± 1.10,50.30 ± 1.10,0.14 ± 0.01
5,flex,AC_CRE,DM-DRL,5.23 ± 0.05,0.00,49.12 ± 1.44,49.12 ± 1.44,0.15 ± 0.01
6,flex,AC_CRE,DRL-DRL,4.00,0.00,99.97 ± 0.06,99.97 ± 0.06,0.48 ± 0.00
7,flex_kl_reg,AC_CRE,DRL-DRL,4.10 ± 0.02,0.00,77.79 ± 1.17,77.79 ± 1.17,0.27 ± 0.01
8,noflex_matrix,AC_CRE,DRL-DRL,1.13 ± 0.02,89.90 ± 1.13,23.37 ± 1.42,13.74 ± 1.50,0.68 ± 0.02
9,no_flex_kl_reg,AC_CRE,DRL-DRL,3.34 ± 0.05,9.47 ± 1.00,56.41 ± 0.79,49.95 ± 0.99,0.18 ± 0.01


### End-activity distribution (mean share over runs, %) — all conditions

In [8]:
# Mean share per end activity; runs where an activity never ends a case count as 0.
full = (ends.pivot_table(index=["condition", "log", "policy", "run_id"], columns="end_activity", values="share", fill_value=0)
            .groupby(["condition", "log", "policy"]).mean())
for label in LOGS.values():
    sub = full.xs(label, level="log").dropna(axis=1, how="all")
    sub = sub.loc[:, (sub > 0).any()]
    sub = sub[sub.loc[("original", "Original (train)")].sort_values(ascending=False).index.union(sub.columns, sort=False)]
    sub = sub.reindex(sorted(sub.index, key=lambda ix: (cond_order.index(ix[0]), pol_order.index(ix[1]))))
    print(f"\n=== {label} ===")
    display((sub * 100).round(1)) if "display" in globals() else print((sub * 100).round(1).to_string())


=== AC_CRE ===
end_activity                     Notificacion estudiante cancelacion soli  Homologacion por grupo de cursos  Evaluacion curso  Visto Bueno Cierre Proceso  Validar solicitud / pre-homologacion  Transferir creditos homologables  Cancelar curso  Radicar Solicitud Homologacion  Transferir Creditos  Cancelar Solicitud  Validar solicitud  Avanzar recepcion documentos  Recepcion de documentos  Validacion final
condition      policy                                                                                                                                                                                                                                                                                                                                                                                                 
original       Original (train)                                      51.5                              17.8               8.0                         6.8                 

In [9]:
# For reference: dominant end activity of DRL-DRL (flex) vs. the a-priori focus activity
dom = (ends[(ends.condition == PAPER_CONDITION) & (ends.policy == "DRL-DRL")]
       .groupby(["log", "end_activity"])["share"].mean().groupby(level=0).idxmax())
dom

log
AC_CRE      (AC_CRE, Notificacion estudiante cancelacion soli)
BPIC12_W                        (BPIC12_W, W_Afhandelen leads)
BPIC17_W                       (BPIC17_W, W_Call after offers)
Name: share, dtype: object

## Paper table — flex condition

In [10]:
paper = agg[((agg.condition == "original") & (agg.policy == "Original (train)")) |
            ((agg.condition == PAPER_CONDITION) & agg.policy.isin(PAPER_POLICIES))].copy()
paper.loc[paper.policy == "Original (train)", "policy"] = "Original"
p_order = ["Original"] + PAPER_POLICIES
paper["_l"] = paper["log"].map(list(LOGS.values()).index)
paper["_p"] = paper["policy"].map(p_order.index)
paper = paper.sort_values(["_l", "_p"]).drop(columns=["_l", "_p"])

paper_view = fmt(paper, ["events_per_case", "canceled_share", "focus_end_share", "tvd_end"], pct=PCT).drop(columns="condition")
paper_view

,log,policy,events_per_case,canceled_share,focus_end_share,tvd_end
0,AC_CRE,Original,4.89,54.52,51.51,0.00
2,AC_CRE,DM-RR,5.18 ± 0.06,50.98 ± 2.28,50.98 ± 2.28,0.14 ± 0.01
5,AC_CRE,DM-DRL,5.23 ± 0.05,49.12 ± 1.44,49.12 ± 1.44,0.15 ± 0.01
6,AC_CRE,DRL-DRL,4.00,99.97 ± 0.06,99.97 ± 0.06,0.48 ± 0.00
10,BPIC12_W,Original,5.39,—,33.27,0.00
12,BPIC12_W,DM-RR,6.84 ± 0.07,—,34.07 ± 0.60,0.39 ± 0.00
15,BPIC12_W,DM-DRL,6.91 ± 0.07,—,33.80 ± 0.75,0.39 ± 0.00
16,BPIC12_W,DRL-DRL,1.06 ± 0.00,—,97.90 ± 0.19,0.65 ± 0.00
20,BPIC17_W,Original,7.21,—,47.47,0.00
22,BPIC17_W,DM-RR,7.36 ± 0.04,—,47.73 ± 0.43,0.01 ± 0.00


In [11]:
def cell(row, m, pct=False, digits=2):
    a, b = row.get(f"{m}_mean"), row.get(f"{m}_ci95")
    if a is None or pd.isna(a):
        return "--"
    k = 100 if pct else 1
    d = 1 if pct else digits
    return f"{a*k:.{d}f}" if (b is None or pd.isna(b) or b == 0) else f"{a*k:.{d}f} $\\pm$ {b*k:.{d}f}"

focus_tex = {label: FOCUS_END[log].replace("_", "\\_") for log, label in LOGS.items()}
focus_tex["AC_CRE"] = "Notif. cancel."  # long Spanish name abbreviated
lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{Case outcomes (flex configuration). Events/case: mean trace length. Canceled: share of cases ending in a cancellation activity (AC\_CRE only; the BPIC logs contain only work items, so no explicit outcome is observable). Focus end: share of cases ending in the most frequent end activity of the original log (in parentheses). TVD: total variation distance between the end-activity distribution and that of the original log. Mean $\pm$ 95\% CI over 10 runs.}",
    r"\label{tab:case-outcomes}",
    r"\small",
    r"\begin{tabular}{llcccc}",
    r"\toprule",
    r"Log (focus end) & Policy & Events/case & Canceled (\%) & Focus end (\%) & TVD \\",
    r"\midrule",
]
for i, (label, grp) in enumerate(paper.groupby("log", sort=False)):
    if i:
        lines.append(r"\midrule")
    for j, (_, r) in enumerate(grp.iterrows()):
        first = f"{label.replace('_', chr(92) + '_')} ({focus_tex[label]})" if j == 0 else ""
        lines.append(f"{first} & {r.policy} & {cell(r, 'events_per_case')} & {cell(r, 'canceled_share', True)} & "
                     f"{cell(r, 'focus_end_share', True)} & {cell(r, 'tvd_end')} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
tex = "\n".join(lines)
(OUT_DIR / "case_outcomes_paper.tex").write_text(tex, encoding="utf-8")
print(tex)

\begin{table}[t]
\centering
\caption{Case outcomes (flex configuration). Events/case: mean trace length. Canceled: share of cases ending in a cancellation activity (AC\_CRE only; the BPIC logs contain only work items, so no explicit outcome is observable). Focus end: share of cases ending in the most frequent end activity of the original log (in parentheses). TVD: total variation distance between the end-activity distribution and that of the original log. Mean $\pm$ 95\% CI over 10 runs.}
\label{tab:case-outcomes}
\small
\begin{tabular}{llcccc}
\toprule
Log (focus end) & Policy & Events/case & Canceled (\%) & Focus end (\%) & TVD \\
\midrule
AC\_CRE (Notif. cancel.) & Original & 4.89 & 54.5 & 51.5 & 0.00 \\
 & DM-RR & 5.18 $\pm$ 0.06 & 51.0 $\pm$ 2.3 & 51.0 $\pm$ 2.3 & 0.14 $\pm$ 0.01 \\
 & DM-DRL & 5.23 $\pm$ 0.05 & 49.1 $\pm$ 1.4 & 49.1 $\pm$ 1.4 & 0.15 $\pm$ 0.01 \\
 & DRL-DRL & 4.00 & 100.0 $\pm$ 0.1 & 100.0 $\pm$ 0.1 & 0.48 $\pm$ 0.00 \\
\midrule
BPIC12\_W (W\_Afhandelen leads) & 